# AICE Associate end-to-end lab
대상: pandas/Python 기초 학습자 | 선행지식: DataFrame, 분류 기초
목표: EDA→전처리→모델→평가→threshold→재현성을 누수 없이 연결한다.
목차: synthetic data, 품질/그래프, leakage, split, ColumnTransformer, baseline, metric, 개선, 연습, 검증, 확장

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, precision_recall_curve

## 1. Synthetic 데이터
실제 파일 없이 seed를 고정한 고객 이탈 표 데이터를 생성한다.

In [ ]:
rng=np.random.default_rng(42); n=240
df=pd.DataFrame({'age':rng.integers(20,70,n),'monthly_spend':rng.normal(55,18,n).round(2),'region':rng.choice(['east','west','south'],n),'support_calls':rng.poisson(2,n)})
logit=-2+.035*df.age+.035*df.support_calls+.018*df.monthly_spend+(df.region=='south')*.45
df['churn']=(rng.random(n)<1/(1+np.exp(-logit))).astype(int); df.loc[rng.choice(n,12,replace=False),'monthly_spend']=np.nan; df.head()

In [ ]:
print(df.shape); print(df.dtypes); print(df.isna().sum()); print(df.groupby('region')['churn'].agg(['count','mean']))
fig,ax=plt.subplots(1,2,figsize=(9,3)); df.monthly_spend.hist(ax=ax[0]); df.groupby('region').churn.mean().plot.bar(ax=ax[1]); plt.tight_layout()

## 2. Leakage 방지
테스트 통계나 예측 시점 이후 변수는 학습에 섞지 않는다. split 후 Pipeline 안에서 fit한다.

In [ ]:
X=df.drop(columns='churn'); y=df.churn
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
print(X_train.shape,X_test.shape,y_train.mean(),y_test.mean())

In [ ]:
num=['age','monthly_spend','support_calls']; cat=['region']
preprocess=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])

## 3. Baseline과 평가
동일 split에서 단순 기준선을 먼저 만들고 업무 오류 비용에 맞는 metric을 확인한다.

In [ ]:
model=Pipeline([('preprocess',preprocess),('classifier',LogisticRegression(max_iter=500,random_state=42))]); model.fit(X_train,y_train)
proba=model.predict_proba(X_test)[:,1]; pred=(proba>=.5).astype(int); print(classification_report(y_test,pred,zero_division=0))

In [ ]:
print('ROC-AUC',round(roc_auc_score(y_test,proba),3)); print(confusion_matrix(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred); plt.tight_layout()

## 4. Threshold trade-off
recall을 높이면 놓침은 줄지만 false positive가 늘 수 있다. 업무 비용과 함께 선택한다.

In [ ]:
precision,recall,thresholds=precision_recall_curve(y_test,proba)
plt.plot(thresholds,precision[:-1],label='precision'); plt.plot(thresholds,recall[:-1],label='recall'); plt.legend(); plt.tight_layout()
valid=np.where(recall[:-1]>=.70)[0]; chosen_threshold=float(thresholds[valid[-1]]) if len(valid) else .5; adjusted=(proba>=chosen_threshold).astype(int); print(chosen_threshold)

## 5. Exercise
TODO: support_calls 단일 특성 baseline AUC와 target recall=.80 threshold를 계산하라. region별 오류도 비교하라.

In [ ]:
print('TODO: implement the exercise before reading the answer')

In [ ]:
single=Pipeline([('impute',SimpleImputer(strategy='median')),('clf',LogisticRegression(max_iter=500,random_state=42))]); single.fit(X_train[['support_calls']],y_train)
single_auc=roc_auc_score(y_test,single.predict_proba(X_test[['support_calls']])[:,1]); assert 0<=single_auc<=1 and chosen_threshold>0; print('answer check PASS',round(single_auc,3))

## 6. 흔한 실수·확장
전체 데이터 fit, accuracy만 사용, seed 생략, 상관=인과 해석, 실패 집단 숨기기를 피한다. RandomForest·교차검증·PR-AUC·비용 기반 threshold로 확장한다.

In [ ]:
again=Pipeline([('preprocess',preprocess),('classifier',LogisticRegression(max_iter=500,random_state=42))]); again.fit(X_train,y_train)
assert np.allclose(proba,again.predict_proba(X_test)[:,1]); print('reproducibility check: PASS')